In [ ]:
!pip install ogx_client

In [38]:
from ogx_client import OgxClient
import rich

In [ ]:
# Configuration
OGX_CONNECTION_URL = "http://ogxserver-service.llama.svc.cluster.local:8321"
POLICY_FILE = "data/return-policy.txt"

In [40]:
# Initialize OGX client
client = OgxClient(base_url=OGX_CONNECTION_URL)

In [41]:
# List available models
models = client.models.list()
rich.print(models)

ListModelsResponse(
    data=[
        Model(
            id='vllm-inference/llama-32-3b-instruct',
            created=1781151488,
            owned_by='ogx',
            custom_metadata={
                'model_type': 'llm',
                'provider_id': 'vllm-inference',
                'provider_resource_id': 'llama-32-3b-instruct'
            },
            object='model'
        ),
        Model(
            id='sentence-transformers/nomic-ai/nomic-embed-text-v1.5',
            created=1781151488,
            owned_by='ogx',
            custom_metadata={
                'model_type': 'embedding',
                'provider_id': 'sentence-transformers',
                'provider_resource_id': 'nomic-ai/nomic-embed-text-v1.5',
                'embedding_dimension': 768
            },
            object='model'
        )
    ],
    object='list'
)

In [42]:
# Checking if any vector db present
rich.print(client.vector_stores.list())

SyncOpenAICursorPage[VectorStore](data=[], has_more=False, last_id='', object='list', first_id='')

In [ ]:
# Extract LLM and embedding model details
llm_model = next(
    m for m in models.data
    if m.custom_metadata.get("model_type") == "llm"
)

# Using specifically sentence-transformers because customized the config to use this inline model
embedding_model = next(
    m for m in models.data
    if m.custom_metadata.get("model_type") == "embedding" and m.custom_metadata.get("provider_id") == "sentence-transformers"
)

model_id = llm_model.id
embedding_model_id = embedding_model.id
embedding_dimension = embedding_model.custom_metadata["embedding_dimension"]

print(f"LLM Model: {model_id}")
print(f"Embedding Model: {embedding_model_id}")
print(f"Embedding Dimension: {embedding_dimension}")

### Verify Vector Store files

In [43]:
# Create vector store with inline FAISS
vector_store = client.vector_stores.create(
    name="techmart_policy_store_inline_faiss",
    extra_body={
        "embedding_model": embedding_model_id,
        "embedding_dimension": embedding_dimension,
        "provider_id": "faiss",
    },
)

vector_store_id = vector_store.id
print(f"Created vector store: {vector_store_id}")

Created vector store: vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6


In [44]:
rich.print(client.vector_stores.list())

SyncOpenAICursorPage[VectorStore](
    data=[
        VectorStore(
            id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6',
            created_at=1781151504,
            file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0),
            status='completed',
            expires_after=None,
            expires_at=None,
            last_active_at=1781151504,
            metadata={
                'provider_id': 'faiss',
                'provider_vector_store_id': 'vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6',
                'embedding_model': 'sentence-transformers/nomic-ai/nomic-embed-text-v1.5',
                'embedding_dimension': '768'
            },
            name='techmart_policy_store_inline_faiss',
            object='vector_store',
            usage_bytes=0
        )
    ],
    has_more=False,
    last_id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6',
    object='list',
    first_id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6'
)

In [45]:
# Upload return policy document
with open(POLICY_FILE, "rb") as f:
    file_info = client.files.create(
        file=("return-policy.txt", f),
        purpose="assistants",
    )

print(f"Uploaded file: {file_info.id}")

Uploaded file: file-6d10057cb1304e6bae7fd20d8805370b


In [46]:
# Upload return policy document
with open(POLICY_FILE, "rb") as f:
    file_info2 = client.files.create(
        file=("return-policy.txt", f),
        purpose="assistants",
    )

print(f"Uploaded file: {file_info2.id}")

Uploaded file: file-c9181add5fcb41898b7bdd354593c9af


In [47]:
# Add file to vector store with chunking strategy
vector_store_file = client.vector_stores.files.create(
    vector_store_id=vector_store_id,
    file_id=file_info.id,
    chunking_strategy={
        "type": "static",
        "static": {
            "max_chunk_size_tokens": 400,
            "chunk_overlap_tokens": 100,
        },
    },
)

rich.print(vector_store_file)

VectorStoreFile(
    id='file-6d10057cb1304e6bae7fd20d8805370b',
    chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(
        static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(
            chunk_overlap_tokens=100,
            max_chunk_size_tokens=400
        ),
        type='static'
    ),
    created_at=1781151515,
    status='completed',
    vector_store_id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6',
    attributes={},
    last_error=None,
    object='vector_store.file',
    usage_bytes=0
)

In [48]:
# Add file to vector store with chunking strategy
vector_store_file_2 = client.vector_stores.files.create(
    vector_store_id=vector_store_id,
    file_id=file_info2.id,
    chunking_strategy={
        "type": "static",
        "static": {
            "max_chunk_size_tokens": 400,
            "chunk_overlap_tokens": 100,
        },
    },
)

rich.print(vector_store_file_2)

VectorStoreFile(
    id='file-c9181add5fcb41898b7bdd354593c9af',
    chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(
        static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(
            chunk_overlap_tokens=100,
            max_chunk_size_tokens=400
        ),
        type='static'
    ),
    created_at=1781151517,
    status='completed',
    vector_store_id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6',
    attributes={},
    last_error=None,
    object='vector_store.file',
    usage_bytes=0
)

In [49]:
rich.print(client.vector_stores.list())

SyncOpenAICursorPage[VectorStore](
    data=[
        VectorStore(
            id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6',
            created_at=1781151504,
            file_counts=FileCounts(cancelled=0, completed=2, failed=0, in_progress=0, total=2),
            status='completed',
            expires_after=None,
            expires_at=None,
            last_active_at=1781151504,
            metadata={
                'provider_id': 'faiss',
                'provider_vector_store_id': 'vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6',
                'embedding_model': 'sentence-transformers/nomic-ai/nomic-embed-text-v1.5',
                'embedding_dimension': '768'
            },
            name='techmart_policy_store_inline_faiss',
            object='vector_store',
            usage_bytes=0
        )
    ],
    has_more=False,
    last_id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6',
    object='list',
    first_id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6'
)

In [50]:
# Verify file is completed
files = client.vector_stores.files.list(vector_store_id)
rich.print(files)

SyncOpenAICursorPage[VectorStoreFile](
    data=[
        VectorStoreFile(
            id='file-c9181add5fcb41898b7bdd354593c9af',
            chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(
                static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(
                    chunk_overlap_tokens=100,
                    max_chunk_size_tokens=400
                ),
                type='static'
            ),
            created_at=1781151517,
            status='completed',
            vector_store_id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6',
            attributes={},
            last_error=None,
            object='vector_store.file',
            usage_bytes=0
        ),
        VectorStoreFile(
            id='file-6d10057cb1304e6bae7fd20d8805370b',
            chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(
                static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(
                    chunk_overlap_tokens=100,
                    max_chunk_size_tokens=400
                ),
                type='static'
            ),
            created_at=1781151515,
            status='completed',
            vector_store_id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6',
            attributes={},
            last_error=None,
            object='vector_store.file',
            usage_bytes=0
        )
    ],
    has_more=False,
    last_id='file-6d10057cb1304e6bae7fd20d8805370b',
    object='list',
    first_id='file-c9181add5fcb41898b7bdd354593c9af'
)

In [23]:
# Inspects a specific file inside (in our case first file) the store to check if processing is 'completed'
file_metadata = client.vector_stores.files.retrieve(
    vector_store_id=vector_store_id,
    file_id=file_info.id
)
# Use attributes that are standard to VectorStoreFile objects
print(f"File ID: {file_metadata.id}")
print(f"Processing Status: {file_metadata.status}")
rich.print(file_metadata)

File ID: file-d9cced1894474c91bac85ac913139909
Processing Status: completed


VectorStoreFile(
    id='file-d9cced1894474c91bac85ac913139909',
    chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(
        static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(
            chunk_overlap_tokens=100,
            max_chunk_size_tokens=400
        ),
        type='static'
    ),
    created_at=1781092068,
    status='completed',
    vector_store_id='vs_c7aec176-fd0d-47e3-b79d-b46cb6fba945',
    attributes={},
    last_error=None,
    object='vector_store.file',
    usage_bytes=0
)

In [24]:
# Call the content method
file_chunks_content = client.vector_stores.files.content(
    vector_store_id=vector_store_id,
    file_id=file_info.id
)

print("--- Extracted File Content ---")
# Loop through the rows/chunks stored inside the data attribute
for item in file_chunks_content.data:
    # Try accessing the text payload (typically under .content, .text, or .chunk)
    if hasattr(item, "content"):
        print(item.content)
    elif hasattr(item, "text"):
        print(item.text)
    else:
        # Fallback if the item itself is a dictionary or direct string
        print(item)

--- Extracted File Content ---
TechMart Return and Refund Policy
Shipping Information:
- Standard Shipping: 3-5 business days (Free on orders over $50)
- Express Shipping: 1-2 business days ($15.99)
- Overnight Shipping: Next business day ($29.99, order before 2 PM EST)
- Orders are processed within 24 hours on business days
- Tracking number sent via email once shipped
Return Time Limits:
- Standard items can be returned within 30 days of delivery
- Electronics must be returned within 15 days of delivery
- Opened software and personalized items cannot be returned
Return Conditions:
Items must be in original condition with original packaging intact. All accessories, manuals, and tags must be included. Items showing signs of use may receive partial refund or be rejected.
How to Return an Item:
1. Log into your TechMart account
2. Go to My Orders and select the order
3. Click Return Item and choose a reason
4. You will receive a return label via email within 24 hours
5. Pack the item sec

In [25]:
# Update file attributes/metadata inside the vector store
updated_file = client.vector_stores.files.update(
    file_id=file_info.id,
    vector_store_id=vector_store_id,
    attributes={
        "department": "return-support",
        "last_updated_by": "admin",
        "version": "2.0"
    }
)

print(f"File updated! New attribute state: {updated_file}")

File updated! New attribute state: VectorStoreFile(id='file-d9cced1894474c91bac85ac913139909', chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(chunk_overlap_tokens=100, max_chunk_size_tokens=400), type='static'), created_at=1781092068, status='completed', vector_store_id='vs_c7aec176-fd0d-47e3-b79d-b46cb6fba945', attributes={'department': 'return-support', 'last_updated_by': 'admin', 'version': '2.0'}, last_error=None, object='vector_store.file', usage_bytes=0)


In [26]:
# Check updated attributes
file_metadata = client.vector_stores.files.retrieve(
    vector_store_id=vector_store_id,
    file_id=file_info.id
)
# Use attributes that are standard to VectorStoreFile objects
print(f"File ID: {file_metadata.id}")
rich.print(file_metadata)

File ID: file-d9cced1894474c91bac85ac913139909


VectorStoreFile(
    id='file-d9cced1894474c91bac85ac913139909',
    chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(
        static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(
            chunk_overlap_tokens=100,
            max_chunk_size_tokens=400
        ),
        type='static'
    ),
    created_at=1781092068,
    status='completed',
    vector_store_id='vs_c7aec176-fd0d-47e3-b79d-b46cb6fba945',
    attributes={'department': 'return-support', 'last_updated_by': 'admin', 'version': '2.0'},
    last_error=None,
    object='vector_store.file',
    usage_bytes=0
)

In [27]:
# Check attributes of second file
file_metadata = client.vector_stores.files.retrieve(
    vector_store_id=vector_store_id,
    file_id=file_info2.id
)
# Use attributes that are standard to VectorStoreFile objects
print(f"File ID: {file_metadata.id}")
rich.print(file_metadata)

File ID: file-e7188000612044e09f202f1c822047c2


VectorStoreFile(
    id='file-e7188000612044e09f202f1c822047c2',
    chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(
        static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(
            chunk_overlap_tokens=100,
            max_chunk_size_tokens=400
        ),
        type='static'
    ),
    created_at=1781092081,
    status='completed',
    vector_store_id='vs_c7aec176-fd0d-47e3-b79d-b46cb6fba945',
    attributes={},
    last_error=None,
    object='vector_store.file',
    usage_bytes=0
)

In [29]:
rich.print(client.vector_stores.list())

SyncOpenAICursorPage[VectorStore](
    data=[
        VectorStore(
            id='vs_c7aec176-fd0d-47e3-b79d-b46cb6fba945',
            created_at=1781091919,
            file_counts=FileCounts(cancelled=0, completed=2, failed=0, in_progress=0, total=2),
            status='completed',
            expires_after=None,
            expires_at=None,
            last_active_at=1781091919,
            metadata={
                'provider_id': 'faiss',
                'provider_vector_store_id': 'vs_c7aec176-fd0d-47e3-b79d-b46cb6fba945',
                'embedding_model': 'sentence-transformers/nomic-ai/nomic-embed-text-v1.5',
                'embedding_dimension': '768'
            },
            name='techmart_policy_store_inline_faiss',
            object='vector_store',
            usage_bytes=0
        )
    ],
    has_more=False,
    last_id='vs_c7aec176-fd0d-47e3-b79d-b46cb6fba945',
    object='list',
    first_id='vs_c7aec176-fd0d-47e3-b79d-b46cb6fba945'
)

In [53]:
file_metadata = client.vector_stores.files.delete(
    vector_store_id=vector_store_id,
    file_id=file_info.id
)

In [ ]:
file_metadata = client.vector_stores.files.delete(
    vector_store_id=vector_store_id,
    file_id=file_info2.id
)

In [54]:
rich.print(client.vector_stores.files.list(vector_store_id=vector_store_id))

SyncOpenAICursorPage[VectorStoreFile](data=[], has_more=False, last_id='', object='list', first_id='')

In [57]:
rich.print(client.vector_stores.list())

SyncOpenAICursorPage[VectorStore](
    data=[
        VectorStore(
            id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6',
            created_at=1781151504,
            file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0),
            status='completed',
            expires_after=None,
            expires_at=None,
            last_active_at=1781151504,
            metadata={
                'provider_id': 'faiss',
                'provider_vector_store_id': 'vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6',
                'embedding_model': 'sentence-transformers/nomic-ai/nomic-embed-text-v1.5',
                'embedding_dimension': '768'
            },
            name='techmart_policy_store_inline_faiss',
            object='vector_store',
            usage_bytes=0
        )
    ],
    has_more=False,
    last_id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6',
    object='list',
    first_id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6'
)

### Verify Vector Store files batch

In [55]:
# Upload return policy document
with open(POLICY_FILE, "rb") as f:
    file_info1 = client.files.create(
        file=("return-policy.txt", f),
        purpose="assistants",
    )

print(f"Uploaded file: {file_info1.id}")

# Upload return policy document
with open(POLICY_FILE, "rb") as f:
    file_info2 = client.files.create(
        file=("return-policy.txt", f),
        purpose="assistants",
    )

print(f"Uploaded file: {file_info2.id}")

Uploaded file: file-dfadb4defa8248d299ff654a2522d5f0
Uploaded file: file-8426c9a56a734000b2c058f572f4fc7e


In [58]:
batch = client.vector_stores.file_batches.create(
    vector_store_id=vector_store_id,
    file_ids=[file_info1.id, file_info2.id],
)
print(batch)

VectorStoreFileBatches(id='batch_210f6b8b-845a-4ee0-b41b-52c075080bff', created_at=1781152044, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=2, total=2), status='in_progress', vector_store_id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6', object='vector_store.files_batch')


In [61]:
batch = client.vector_stores.file_batches.retrieve(
    vector_store_id=vector_store_id,
    batch_id=batch.id,
)
print(batch)

VectorStoreFileBatches(id='batch_210f6b8b-845a-4ee0-b41b-52c075080bff', created_at=1781152044, file_counts=FileCounts(cancelled=0, completed=2, failed=0, in_progress=0, total=2), status='completed', vector_store_id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6', object='vector_store.files_batch')


In [60]:
files = client.vector_stores.file_batches.list_files(
    vector_store_id=vector_store_id,
    batch_id=batch.id,
)
for f in files:
    print(f.id)

file-dfadb4defa8248d299ff654a2522d5f0
file-8426c9a56a734000b2c058f572f4fc7e


In [78]:
## creating a batch to verify cancel
batch = client.vector_stores.file_batches.create(
    vector_store_id=vector_store_id,
    file_ids=[file_info1.id, file_info2.id],
)
print(batch)

batch = client.vector_stores.file_batches.cancel(
    vector_store_id=vector_store_id,
    batch_id=batch.id
)
print(batch)

VectorStoreFileBatches(id='batch_305e27dc-032c-4150-8b40-420586c7c575', created_at=1781153249, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=2, total=2), status='in_progress', vector_store_id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6', object='vector_store.files_batch')
VectorStoreFileBatches(id='batch_305e27dc-032c-4150-8b40-420586c7c575', created_at=1781153249, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=2, total=2), status='cancelled', vector_store_id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6', object='vector_store.files_batch')


In [67]:
vector_store = client.vector_stores.retrieve(vector_store_id)
print(vector_store)

VectorStore(id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6', created_at=1781151504, file_counts=FileCounts(cancelled=0, completed=2, failed=0, in_progress=0, total=2), status='completed', expires_after=None, expires_at=None, last_active_at=1781151504, metadata={'provider_id': 'faiss', 'provider_vector_store_id': 'vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6', 'embedding_model': 'sentence-transformers/nomic-ai/nomic-embed-text-v1.5', 'embedding_dimension': '768'}, name='techmart_policy_store_inline_faiss', object='vector_store', usage_bytes=0)


In [69]:
vector_store = client.vector_stores.update(
    vector_store_id,
    name="my-vector-store",
)
rich.print(vector_store)

VectorStore(
    id='vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6',
    created_at=1781151504,
    file_counts=FileCounts(cancelled=0, completed=2, failed=0, in_progress=0, total=2),
    status='completed',
    expires_after=None,
    expires_at=None,
    last_active_at=1781153089,
    metadata={
        'provider_id': 'faiss',
        'provider_vector_store_id': 'vs_c9c4d4a9-24f9-4f6f-bc2a-df35b9c727b6',
        'embedding_model': 'sentence-transformers/nomic-ai/nomic-embed-text-v1.5',
        'embedding_dimension': '768'
    },
    name='my-vector-store',
    object='vector_store',
    usage_bytes=0
)

In [98]:
rich.print(client.vector_stores.delete(vector_store_id))

VectorStoreDeleteResponse(
    id='vs_7925294a-35f4-4114-9933-c846c24682f0',
    deleted=True,
    object='vector_store.deleted'
)

## Verify Vector stores API

In [99]:
# Checking if any vector db present
rich.print(client.vector_stores.list())

SyncOpenAICursorPage[VectorStore](data=[], has_more=False, last_id='', object='list', first_id='')

In [100]:
# Create vector store with inline FAISS
vector_store = client.vector_stores.create(
    name="techmart_policy_store_inline_faiss",
    extra_body={
        "embedding_model": embedding_model_id,
        "embedding_dimension": embedding_dimension,
        "provider_id": "faiss",
    },
)

vector_store_id = vector_store.id
print(f"Created vector store: {vector_store_id}")

Created vector store: vs_2aaa248d-6c98-4ee4-9a09-67e30ae97d79


In [101]:
rich.print(client.vector_stores.list())

SyncOpenAICursorPage[VectorStore](
    data=[
        VectorStore(
            id='vs_2aaa248d-6c98-4ee4-9a09-67e30ae97d79',
            created_at=1781163676,
            file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0),
            status='completed',
            expires_after=None,
            expires_at=None,
            last_active_at=1781163676,
            metadata={
                'provider_id': 'faiss',
                'provider_vector_store_id': 'vs_2aaa248d-6c98-4ee4-9a09-67e30ae97d79',
                'embedding_model': 'sentence-transformers/nomic-ai/nomic-embed-text-v1.5',
                'embedding_dimension': '768'
            },
            name='techmart_policy_store_inline_faiss',
            object='vector_store',
            usage_bytes=0
        )
    ],
    has_more=False,
    last_id='vs_2aaa248d-6c98-4ee4-9a09-67e30ae97d79',
    object='list',
    first_id='vs_2aaa248d-6c98-4ee4-9a09-67e30ae97d79'
)

In [102]:
# Upload return policy document
with open(POLICY_FILE, "rb") as f:
    file_info = client.files.create(
        file=("return-policy.txt", f),
        purpose="assistants",
    )

print(f"Uploaded file: {file_info.id}")

Uploaded file: file-62924ce0337d4aed9cfc0077bd13e731


In [103]:
# Add file to vector store with chunking strategy
vector_store_file = client.vector_stores.files.create(
    vector_store_id=vector_store_id,
    file_id=file_info.id,
    chunking_strategy={
        "type": "static",
        "static": {
            "max_chunk_size_tokens": 400,
            "chunk_overlap_tokens": 100,
        },
    },
)

rich.print(vector_store_file)

VectorStoreFile(
    id='file-62924ce0337d4aed9cfc0077bd13e731',
    chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(
        static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(
            chunk_overlap_tokens=100,
            max_chunk_size_tokens=400
        ),
        type='static'
    ),
    created_at=1781163733,
    status='completed',
    vector_store_id='vs_2aaa248d-6c98-4ee4-9a09-67e30ae97d79',
    attributes={},
    last_error=None,
    object='vector_store.file',
    usage_bytes=0
)

In [104]:
rich.print(client.vector_stores.list())

SyncOpenAICursorPage[VectorStore](
    data=[
        VectorStore(
            id='vs_2aaa248d-6c98-4ee4-9a09-67e30ae97d79',
            created_at=1781163676,
            file_counts=FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1),
            status='completed',
            expires_after=None,
            expires_at=None,
            last_active_at=1781163676,
            metadata={
                'provider_id': 'faiss',
                'provider_vector_store_id': 'vs_2aaa248d-6c98-4ee4-9a09-67e30ae97d79',
                'embedding_model': 'sentence-transformers/nomic-ai/nomic-embed-text-v1.5',
                'embedding_dimension': '768'
            },
            name='techmart_policy_store_inline_faiss',
            object='vector_store',
            usage_bytes=0
        )
    ],
    has_more=False,
    last_id='vs_2aaa248d-6c98-4ee4-9a09-67e30ae97d79',
    object='list',
    first_id='vs_2aaa248d-6c98-4ee4-9a09-67e30ae97d79'
)

In [105]:
# Retrive the specific vector store details
vector_store = client.vector_stores.retrieve(vector_store_id)
rich.print(vector_store)

VectorStore(
    id='vs_2aaa248d-6c98-4ee4-9a09-67e30ae97d79',
    created_at=1781163676,
    file_counts=FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1),
    status='completed',
    expires_after=None,
    expires_at=None,
    last_active_at=1781163676,
    metadata={
        'provider_id': 'faiss',
        'provider_vector_store_id': 'vs_2aaa248d-6c98-4ee4-9a09-67e30ae97d79',
        'embedding_model': 'sentence-transformers/nomic-ai/nomic-embed-text-v1.5',
        'embedding_dimension': '768'
    },
    name='techmart_policy_store_inline_faiss',
    object='vector_store',
    usage_bytes=0
)

In [106]:
# update the name of vector store
vector_store = client.vector_stores.update(
    vector_store_id,
    name="techmart_inline_faiss",
)
rich.print(vector_store)

VectorStore(
    id='vs_2aaa248d-6c98-4ee4-9a09-67e30ae97d79',
    created_at=1781163676,
    file_counts=FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1),
    status='completed',
    expires_after=None,
    expires_at=None,
    last_active_at=1781163851,
    metadata={
        'provider_id': 'faiss',
        'provider_vector_store_id': 'vs_2aaa248d-6c98-4ee4-9a09-67e30ae97d79',
        'embedding_model': 'sentence-transformers/nomic-ai/nomic-embed-text-v1.5',
        'embedding_dimension': '768'
    },
    name='techmart_inline_faiss',
    object='vector_store',
    usage_bytes=0
)

In [107]:
client.vector_stores.delete(vector_store_id)

VectorStoreDeleteResponse(id='vs_2aaa248d-6c98-4ee4-9a09-67e30ae97d79', deleted=True, object='vector_store.deleted')

## Verify Vector store insert and Query

### Insert embeddings directly

In [109]:
embedding_response = client.embeddings.create(
    input="Docker is a container platform.",
    model="sentence-transformers/nomic-ai/nomic-embed-text-v1.5",
)

rich.print(embedding_response)
embedding = embedding_response.data[0].embedding

print(len(embedding))
print(embedding[:5])

CreateEmbeddingsResponse(
    data=[
        Data(
            embedding=[
                -0.02915007621049881,
                1.8210577964782715,
                -3.214754104614258,
                0.024642471224069595,
                0.887718915939331,
                -0.25228288769721985,
                0.5627367496490479,
                -0.6538081169128418,
                -1.1645920276641846,
                -1.9530694484710693,
                -0.1778656393289566,
                0.3104029595851898,
                2.347486972808838,
                -0.20492404699325562,
                1.0137580633163452,
                -1.5800825357437134,
                -0.2133673131465912,
                -1.1225100755691528,
                1.122177243232727,
                0.8448259830474854,
                -0.8991985321044922,
                -0.617440938949585,
                -0.4214063882827759,
                0.0092322938144207,
                1.6193338632583618,
                1.188408374786377,
                0.44083020091056824,
                0.8461195826530457,
                -1.0802956819534302,
                1.2727148532867432,
                -0.24490581452846527,
                0.14109782874584198,
                0.9565620422363281,
                -0.9025853276252747,
                -0.8537687659263611,
                -2.1288557052612305,
                0.13465376198291779,
                0.3980485796928406,
                -0.11160102486610413,
                -0.3847227394580841,
                -0.4257084131240845,
                0.031180130317807198,
                -0.9316003322601318,
                -1.3438884019851685,
                -0.28444182872772217,
                0.16314992308616638,
                0.5435414910316467,
                -0.5091118812561035,
                0.7375051975250244,
                -0.9448645710945129,
                0.1785559058189392,
                -0.3663513660430908,
                -0.8578971028327942,
                -1.6367594003677368,
                0.7184677124023438,
                1.2250539064407349,
                -1.0633046627044678,
                0.9389930367469788,
                -0.6871117353439331,
                0.6641642451286316,
                0.5999934673309326,
                1.7211875915527344,
                0.5423502922058105,
                3.2253236770629883,
                0.5464985370635986,
                0.5612043738365173,
                0.4790191650390625,
                1.0528944730758667,
                -0.052965305745601654,
                -0.7289183735847473,
                1.9401681423187256,
                0.30874037742614746,
                0.2191431075334549,
                0.5425317287445068,
                0.038836415857076645,
                1.072275996208191,
                -0.9063825607299805,
                -0.14663809537887573,
                -0.5500547885894775,
                0.8140739798545837,
                1.0938606262207031,
                -1.2519431114196777,
                0.9810031056404114,
                0.27604353427886963,
                1.5170937776565552,
                0.35680654644966125,
                -1.084532380104065,
                0.3707316517829895,
                -0.674004077911377,
                1.1418507099151611,
                -0.6493458151817322,
                -0.26317840814590454,
                0.8891597390174866,
                1.6069598197937012,
                -1.7163299322128296,
                0.754340410232544,
                -0.3985174596309662,
                0.6857614517211914,
                0.32009807229042053,
                -0.4894055724143982,
                -0.9327058792114258,
                -0.43070051074028015,
                0.897746741771698,
                -0.34219276905059814,
                0.9331966042518616,
                1.1802898645401,
                0.5887352824211121,
             

768
[-0.02915007621049881, 1.8210577964782715, -3.214754104614258, 0.024642471224069595, 0.887718915939331]


In [111]:
# Create vector store with inline FAISS
vector_store = client.vector_stores.create(
    name="techmart_policy_store_inline_faiss",
    extra_body={
        "embedding_model": embedding_model_id,
        "embedding_dimension": embedding_dimension,
        "provider_id": "faiss",
    },
)

vector_store_id = vector_store.id
print(f"Created vector store: {vector_store_id}")

Created vector store: vs_e5e55174-5368-49d9-9f28-fbcb91970871


In [112]:
import time
import requests

now = int(time.time())

payload = {
    "vector_store_id": vector_store_id,
    "chunks": [
        {
            "chunk_id": "docker-1",
            "content": "Docker is a container platform.",
            "metadata": {},
            "chunk_metadata": {
                "chunk_id": "docker-1",
                "document_id": "docker-doc-1",
                "source": "manual",
                "created_timestamp": now,
                "updated_timestamp": now,
                "chunk_window": "0-100",
                "chunk_tokenizer": "nomic",
                "content_token_count": 5,
                "metadata_token_count": 0
            },
            "embedding": embedding,
            "embedding_model": "sentence-transformers/nomic-ai/nomic-embed-text-v1.5",
            "embedding_dimension": 768
        }
    ]
}

r = requests.post(
    OGX_CONNECTION_URL+"/v1/vector-io/insert",
    json=payload
)

print(r.status_code)
print(r.text)

204



In [113]:
query_result = client.vector_io.query(
    vector_store_id=vector_store_id,
    query="earth",
    params={
        "mode": "vector", # keyword and hybrid not supported for FAISS
        "max_chunks": 1,
        
    },
)

print(query_result)

QueryChunksResponse(chunks=[Chunk(chunk_id='docker-1', chunk_metadata=ChunkChunkMetadata(chunk_id='docker-1', chunk_tokenizer='nomic', chunk_window='0-100', content_token_count=5, created_timestamp=1781164245, document_id='docker-doc-1', metadata_token_count=0, source='manual', updated_timestamp=1781164245), content='Docker is a container platform.', embedding=[-0.02915007621049881, 1.8210577964782715, -3.214754104614258, 0.024642471224069595, 0.887718915939331, -0.25228288769721985, 0.5627367496490479, -0.6538081169128418, -1.1645920276641846, -1.9530694484710693, -0.1778656393289566, 0.3104029595851898, 2.347486972808838, -0.20492404699325562, 1.0137580633163452, -1.5800825357437134, -0.2133673131465912, -1.1225100755691528, 1.122177243232727, 0.8448259830474854, -0.8991985321044922, -0.617440938949585, -0.4214063882827759, 0.0092322938144207, 1.6193338632583618, 1.188408374786377, 0.44083020091056824, 0.8461195826530457, -1.0802956819534302, 1.2727148532867432, -0.24490581452846527,

In [114]:
query_result = client.vector_io.query(
    vector_store_id=vector_store_id,
    query="docker",
    params={
        "mode": "vector", # keyword and hybrid not supported for FAISS
        "max_chunks": 1,
        
    },
)

print(query_result)

QueryChunksResponse(chunks=[Chunk(chunk_id='docker-1', chunk_metadata=ChunkChunkMetadata(chunk_id='docker-1', chunk_tokenizer='nomic', chunk_window='0-100', content_token_count=5, created_timestamp=1781164245, document_id='docker-doc-1', metadata_token_count=0, source='manual', updated_timestamp=1781164245), content='Docker is a container platform.', embedding=[-0.02915007621049881, 1.8210577964782715, -3.214754104614258, 0.024642471224069595, 0.887718915939331, -0.25228288769721985, 0.5627367496490479, -0.6538081169128418, -1.1645920276641846, -1.9530694484710693, -0.1778656393289566, 0.3104029595851898, 2.347486972808838, -0.20492404699325562, 1.0137580633163452, -1.5800825357437134, -0.2133673131465912, -1.1225100755691528, 1.122177243232727, 0.8448259830474854, -0.8991985321044922, -0.617440938949585, -0.4214063882827759, 0.0092322938144207, 1.6193338632583618, 1.188408374786377, 0.44083020091056824, 0.8461195826530457, -1.0802956819534302, 1.2727148532867432, -0.24490581452846527,

In [116]:
rich.print(client.vector_stores.delete(vector_store_id))

VectorStoreDeleteResponse(
    id='vs_e5e55174-5368-49d9-9f28-fbcb91970871',
    deleted=True,
    object='vector_store.deleted'
)